# Быстрый старт RAG Pipeline

Этот ноутбук демонстрирует полный цикл подготовки и запуска RAG-системы:
1. Установка зависимостей (GPU A100 с 25 GB VRAM используется автоматически, при отсутствии GPU всё работает на CPU).
2. Скачивание и квантование выбранной LLM.
3. Построение RAG-индекса на собственных данных.
4. Загрузка вопросов из разных форматов.
5. Инференс и вывод красивой таблицы результатов.
6. Сохранение ответов в нужных форматах.

Все функции снабжены русскими docstring'ами — можно навести курсор и увидеть описание параметров.

**Для Google Colab или удаленной системы:** Запустите секцию "0. Настройка для Colab" перед началом работы.


## 0. Настройка для Colab / Удаленной системы

**Для Google Colab или удаленной системы:** Запустите эту секцию перед началом работы.

**Для локального запуска:** Пропустите эту секцию.


In [1]:
# Для Colab/удаленной системы раскомментируйте:
!git clone https://github.com/phantom2059/baseline_rag.git
!cd baseline_rag && pwd
import sys
sys.path.insert(0, '/content/baseline_rag')
!cd baseline_rag


Cloning into 'baseline_rag'...
remote: Enumerating objects: 63, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 63 (delta 31), reused 52 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (63/63), 106.17 KiB | 15.17 MiB/s, done.
Resolving deltas: 100% (31/31), done.
/content/baseline_rag


In [2]:
import torch

print(f"CUDA доступна: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠ GPU не обнаружен. Включите GPU в настройках Colab или используйте CPU.")


CUDA доступна: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 39.6 GB


## 1. Установка зависимостей (для локального запуска)

**Для Colab:** Используйте секцию 0.2 выше.

**Для локального запуска:** Запустите эту ячейку один раз. Если какой-то пакет уже установлен, команда просто обновит его до актуальной версии.


In [3]:
!pip install -q -U pip
!pip install -q -r /content/baseline_rag/requirements.txt

# Для локального запуска с GPU попробуйте faiss-gpu-cu121
try:
    !pip install -q -U faiss-gpu-cu121
except Exception:
    !pip install -q -U faiss-cpu

print("✓ Зависимости установлены")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 76.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Could not find a version that satisfies the requirement faiss-gpu-cu121 (from versions: none)
ERROR: No matching distribution found for faiss-gpu-cu121
✓ Зависимости установлены


## 2. Загрузка и квантование модели
Укажите нужное имя модели на HuggingFace или прямую ссылку. Функция автоматически скачает веса, применит выбранное квантование и создаст `config.json`.


In [4]:
!pip install autoawq optimum

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
ERROR: Failed to build 'auto-gptq' when getting requirements to build wheel


In [5]:
from model_downloader import download_model

DOWNLOAD_MODEL = True  #False, если модель уже скачана

if DOWNLOAD_MODEL:
    download_model(
        model_name="Qwen/Qwen2.5-Coder-32B-Instruct-GPTQ-Int4",
        model_dir="./model",
        #quantize=False,
        #bits=4,
        #config_path="config.json",
    )
else:
    print("[SKIP] Скачивание модели пропущено — используем существующие веса.")


[MODEL_DOWNLOADER] Модель: Qwen/Qwen2.5-Coder-32B-Instruct-GPTQ-Int4
[MODEL_DOWNLOADER] Целевая папка: /content/model


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00003-of-00005.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/3.98G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/3.48G [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

[MODEL_DOWNLOADER] ✓ Файлы скачаны.
[MODEL_DOWNLOADER] Генерация config.json...
[MODEL_DOWNLOADER] Готово. Конфиг: /content/config.json


## 3. Создание RAG-индекса
Если у вас есть собственные данные, включите флаг `BUILD_RAG` и укажите путь к файлу. Поддерживаются форматы CSV, TSV, XLSX, JSON и TXT. Логика нормализации совпадает с боевым сервисом.

**Альтернатива:** Можно использовать датасеты из Hugging Face — см. секцию 3.1 ниже.


In [ ]:
# !pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 230.9 MB/s  0:00:00


In [ ]:
from utils import convert_hf_dataset

# Пример: конвертация датасета Python кода
# Сначала установите datasets: !pip install datasets

CONVERT_DATASET = False  # включите True для конвертации
DATASET_NAME = "jtatman/python-code-dataset-500k"
OUTPUT_FILE = "python_code.json"

if CONVERT_DATASET:
    convert_hf_dataset(
        dataset_name=DATASET_NAME,
        output_path=OUTPUT_FILE,
        text_column="code",  # укажите название столбца с текстом/кодом
        # title_column="title",  # опционально: столбец с заголовком
        fmt="json",  # или "csv", "tsv"
        # limit=1000
    )
    print(f"[CONVERT] ✓ Датасет сохранён в {OUTPUT_FILE}")
else:
    print("[SKIP] Конвертация датасета пропущена.")

In [ ]:
from rag_builder import create_rag_index

BUILD_RAG = False  # включите True, чтобы пересобрать индекс
DATA_PATH = "data/input.csv"  # или .xlsx/.json/.tsv/.txt
RAG_DIR = "./rag"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L12-v2"

if BUILD_RAG:
    create_rag_index(
        data_path=DATA_PATH,
        output_dir=RAG_DIR,
        embedding_model=EMBEDDING_MODEL,
        chunk_size=700,
        chunk_overlap=120,
        text_column="text",  # название столбца с текстом
        title_column="title",  # название столбца с заголовком (опционально)
    )
else:
    print("[SKIP] Пересборка RAG отключена — используем готовые артефакты.")


## 4. Загрузка вопросов
Укажите файл с вопросами (JSON массив/объект, CSV, TSV, XLSX или TXT). Все форматы определяются автоматически.


In [ ]:
def fix_bom_json(path):
    with open(path, "rb") as f:
        data = f.read()
    bom = b'\xef\xbb\xbf'
    if data.startswith(bom):
        print("BOM найден, удаляю...")
        content = data[len(bom):]
        with open(path + ".bak", "wb") as bak:
            bak.write(data)
            print(f"Резервная копия создана: {path}.bak")
        with open(path, "wb") as f:
            f.write(content)
        print(f"Файл перезаписан без BOM: {path}")
    else:
        print("BOM не найден, изменений не требуется.")

fix_bom_json("/content/data/input.json")


BOM найден, удаляю...
Резервная копия создана: /content/data/input.json.bak
Файл перезаписан без BOM: /content/data/input.json


In [ ]:
from utils import load_questions

INPUT_FILE = "/content/data/input.json"
questions = load_questions(INPUT_FILE)
print(f"[DATA] Загружено {len(questions)} вопросов")
preview_count = min(5, len(questions))
print("[DATA] Примеры:")
for idx in range(preview_count):
    print(f"  {idx+1}. {questions[idx]}")


[DATA] Загружено 60 вопросов
[DATA] Примеры:
  1. Какие были основные последствия Вестфальского мира 1648 года для развития международных отношений в Европе?
  2. В чем заключалась разница между подходами Аристотеля и Платона к теории идей?
  3. Как повлияло изобретение книгопечатания на распространение Реформации в Европе?
  4. Объясните парадокс Эйнштейна-Подольского-Розена и его значение для квантовой механики
  5. В чем разница между алгебраической и трансцендентной топологией?


## 5. Инференс
Создаём экземпляр `FactualModel`, который автоматически загрузит модель и RAG-артефакты. Для ускорения используется GPU A100 (если доступно).


In [ ]:
from factual_model import FactualModel

model = FactualModel()
answers = model.generate_batch(questions, batch_size=8)
print(f"[INFER] Получено ответов: {len(answers)}")


[MODEL] Loading model from /content/model
[MODEL] Loading tokenizer...
[MODEL] ✓ Tokenizer loaded (vocab_size=151665)
[MODEL] CUDA available and accessible
[MODEL] Loading config...
[MODEL] ✓ Config loaded (model_type=qwen2, hidden_size=3584)
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

[MODEL] ✓✓✓ Model loaded successfully (device=cuda:0)
[RAG] Starting RAG initialization...
[RAG] RAG_DIR: /content/rag
[RAG] ✓ SentenceTransformer imported
[RAG] ✓ faiss imported
[RAG] ✓ pyarrow imported
[RAG] Checking files:
[RAG]   RAG_DIR exists: False
[RAG]   RAG_DIR is_dir: False
[RAG]   meta.json: /content/rag/meta.json (exists=False)
[RAG]   index.faiss: /content/rag/index.faiss (exists=False)
[RAG]   chunks.parquet: /content/rag/chunks.parquet (exists=False)
[RAG] ✗ Missing RAG files, checking alternative locations...
[RAG] ✗ Missing RAG files
[RAG] ✗✗✗ RAG initialization FAILED (CRITICAL): RAG requires all files to be present, but missing: meta.json: /content/rag/meta.json, index.faiss: /content/rag/index.faiss, chunks.parquet: /content/rag/chunks.parquet
Traceback (most recent call last):
  File "/content/baseline_rag/factual_model.py", line 386, in _init_rag
    raise FileNotFoundError(f"RAG requires all files to be present, but missing: {', '.join(missing)}")
FileNotFoundEr

[INFER] Получено ответов: 60


## 6. Таблица результатов
Ниже выводится таблица в стиле `saiga_benchmark.ipynb` с первыми примерами ответов.


In [ ]:
from tabulate import tabulate

rows = []
for idx, (q, a) in enumerate(zip(questions, answers), start=1):
    rows.append({
        "#": idx,
        "chars": len(a),
        "question": q if len(q) <= 120 else q[:117] + "...",
        "answer": a,
    })
print(tabulate(rows, headers="keys", tablefmt="github"))


|   # |   chars | question                                                                                                    | answer                                                                                                                                |
|-----|---------|-------------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------------------------------------------------------------------|
|   1 |     110 | Какие были основные последствия Вестфальского мира 1648 года для развития международных отношений в Европе? | Вестфальский мир, заключенный в 1648 году после Тридцатилетней войны, имел значительное значение для развития.                        |
|   2 |       7 | В чем заключалась разница между подходами Аристотеля и Платона к теории идей?                               | Не знаю                                                                         

## 7. Сохранение результатов
Укажите формат и путь. Можно сохранить сразу несколько вариантов (JSON массив, JSON пары, CSV, TSV).


In [ ]:
from utils import save_results

OUTPUT_CSV = "output.csv"

save_results(questions, answers, OUTPUT_CSV, fmt="csv")
print("[SAVE] Результаты записаны в указанные файлы.")


## 9. VibeChat
Минималистичный чат прямо в ноутбуке для «вайбкодинга»:
- Никакого фронта: всё в ячейке, удобно копировать код из ответа
- Гарантированно работает в Google Colab и локально
- По умолчанию без RAG, можно мягко включить фолбэк на RAG при «Не знаю»


In [10]:
!pip install autoawq optimum

  Using cached autoawq-0.2.9.tar.gz (74 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for autoawq: filename=autoawq-0.2.9-py3-none-any.whl size=115198 sha256=17b1095bbcbe0a7c044a0c8af663bf203ac8325470865f2e2d026f744c413d56
  Stored in directory: /root/.cache/pip/wheels/45/1a/7b/7314b3a958454e8ce349f600829a3f0a6a05aeebf987be1e16
Successfully built autoawq


In [11]:
from vibechat import VibeChatSession


_model = globals().get('model')
vc = globals().get('vc')
if vc is None:
    vc = VibeChatSession(model=_model)
    globals()['vc'] = vc

# Пример использования (скопируй и меняй текст запроса):
vc.config(max_new_tokens=4096, rag_fallback=False)
print(vc.ask("Сделай на Python функцию для inplace quicksort."))


AttributeError: 'str' object has no attribute 'vibecode_chat'